# Wossist Dataset Validation

This notebook validates the synthetic datasets used in the **Wossist Insurance Customer Analytics** case study.

The objective is to verify structural integrity, referential integrity, chronological consistency, business rules, customer distributions, pricing logic, marketing attribution, seasonality and renewal behavior.

The notebook loads the published `.csv.gz` files directly from the GitHub repository and is designed to run end-to-end with **Run all**.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load datasets

In [ ]:
BASE_URL = (
    "https://raw.githubusercontent.com/"
    "AntonioPennacchia/customer-analytics/main/"
    "insurance-customer-analytics/dataset/"
)

FILES = {
    "customers": "customers.csv.gz",
    "products": "products.csv.gz",
    "sales_channels": "sales_channels.csv.gz",
    "operators": "operators.csv.gz",
    "campaigns": "campaigns.csv.gz",
    "customer_interactions": "customer_interactions.csv.gz",
    "quotes": "quotes.csv.gz",
    "policies": "policies.csv.gz",
    "renewals": "renewals.csv.gz",
}

dfs = {name: pd.read_csv(BASE_URL + filename) for name, filename in FILES.items()}

customers = dfs["customers"]
products = dfs["products"]
sales_channels = dfs["sales_channels"]
operators = dfs["operators"]
campaigns = dfs["campaigns"]
interactions = dfs["customer_interactions"]
quotes = dfs["quotes"]
policies = dfs["policies"]
renewals = dfs["renewals"]

for name, df in dfs.items():
    print(f"{name:24s} {len(df):>10,} rows | {df.shape[1]:>2} columns")

## 2. Parse dates and helper functions

In [ ]:
date_columns = {
    "customers": ["birth_date", "registration_date"],
    "campaigns": ["start_date", "end_date"],
    "customer_interactions": ["interaction_date"],
    "quotes": ["quote_date"],
    "policies": ["purchase_date", "policy_start_date", "policy_end_date"],
    "renewals": ["renewal_date"],
}

for table_name, cols in date_columns.items():
    for col in cols:
        dfs[table_name][col] = pd.to_datetime(dfs[table_name][col], errors="coerce")

def check(name, condition, details=""):
    status = "PASS" if bool(condition) else "FAIL"
    return {"check": name, "status": status, "details": details}

results = []

## 3. Structural validation

Checks primary-key uniqueness, missing identifiers and expected columns.

In [ ]:
pk_map = {
    "customers": "customer_id",
    "products": "product_id",
    "sales_channels": "sales_channel_id",
    "operators": "operator_id",
    "campaigns": "campaign_id",
    "customer_interactions": "interaction_id",
    "quotes": "quote_id",
    "policies": "policy_id",
    "renewals": "renewal_id",
}

for table_name, pk in pk_map.items():
    df = dfs[table_name]
    results.append(check(
        f"{table_name}: primary key unique",
        df[pk].is_unique,
        f"duplicates={df[pk].duplicated().sum():,}"
    ))
    results.append(check(
        f"{table_name}: primary key not null",
        df[pk].notna().all(),
        f"nulls={df[pk].isna().sum():,}"
    ))

expected_columns = {
    "customers": ["customer_id","first_name","last_name","birth_date","gender","country","region","city","registration_date","acquisition_channel"],
    "products": ["product_id","product_name","business_line","product_type"],
    "sales_channels": ["sales_channel_id","sales_channel_name"],
    "operators": ["operator_id","operator_name","department","role"],
    "campaigns": ["campaign_id","campaign_name","business_line","product_id","objective","marketing_channel","start_date","end_date","campaign_owner_id"],
    "customer_interactions": ["interaction_id","customer_id","campaign_id","interaction_date","interaction_channel","interaction_type","outcome"],
    "quotes": ["quote_id","customer_id","product_id","quote_date","sales_channel_id","campaign_id","estimated_premium"],
    "policies": ["policy_id","customer_id","quote_id","product_id","purchase_date","policy_start_date","policy_end_date","sales_channel_id","campaign_id","premium_amount","destination","trip_duration_days","pet_type","pet_age","property_value","property_size"],
    "renewals": ["renewal_id","policy_id","customer_id","renewal_date","premium_amount"],
}

for table_name, cols in expected_columns.items():
    actual = list(dfs[table_name].columns)
    missing = [c for c in cols if c not in actual]
    extra = [c for c in actual if c not in cols]
    results.append(check(
        f"{table_name}: expected schema",
        not missing and not extra,
        f"missing={missing}; extra={extra}"
    ))

## 4. Referential integrity

In [ ]:
def fk_check(child_df, child_col, parent_df, parent_col, name, allow_null=False):
    vals = child_df[child_col]
    if allow_null:
        vals = vals.dropna()
    bad = ~vals.isin(parent_df[parent_col])
    results.append(check(
        name,
        not bad.any(),
        f"invalid references={bad.sum():,}"
    ))

fk_check(quotes, "customer_id", customers, "customer_id", "quotes -> customers")
fk_check(quotes, "product_id", products, "product_id", "quotes -> products")
fk_check(quotes, "sales_channel_id", sales_channels, "sales_channel_id", "quotes -> sales_channels")
fk_check(quotes, "campaign_id", campaigns, "campaign_id", "quotes -> campaigns", allow_null=True)

fk_check(policies, "customer_id", customers, "customer_id", "policies -> customers")
fk_check(policies, "quote_id", quotes, "quote_id", "policies -> quotes")
fk_check(policies, "product_id", products, "product_id", "policies -> products")
fk_check(policies, "sales_channel_id", sales_channels, "sales_channel_id", "policies -> sales_channels")
fk_check(policies, "campaign_id", campaigns, "campaign_id", "policies -> campaigns", allow_null=True)

fk_check(interactions, "customer_id", customers, "customer_id", "interactions -> customers")
fk_check(interactions, "campaign_id", campaigns, "campaign_id", "interactions -> campaigns", allow_null=True)

fk_check(campaigns, "product_id", products, "product_id", "campaigns -> products")
fk_check(campaigns, "campaign_owner_id", operators, "operator_id", "campaigns -> operators")

fk_check(renewals, "policy_id", policies, "policy_id", "renewals -> policies")
fk_check(renewals, "customer_id", customers, "customer_id", "renewals -> customers")

## 5. Chronological consistency

In [ ]:
# registration <= first purchase
first_purchase = policies.groupby("customer_id")["purchase_date"].min()
cust_dates = customers.set_index("customer_id")[["registration_date"]].join(first_purchase.rename("first_purchase_date"))
bad_reg = cust_dates["first_purchase_date"].notna() & (cust_dates["registration_date"] > cust_dates["first_purchase_date"])
results.append(check(
    "Registration date <= first purchase date",
    not bad_reg.any(),
    f"violations={bad_reg.sum():,}"
))

# quote <= purchase for converted quotes
qp = policies[["policy_id","quote_id","purchase_date"]].merge(
    quotes[["quote_id","quote_date"]], on="quote_id", how="left"
)
bad_qp = qp["quote_date"] > qp["purchase_date"]
results.append(check(
    "Quote date <= policy purchase date",
    not bad_qp.any(),
    f"violations={bad_qp.sum():,}"
))

bad_policy_dates = (policies["policy_start_date"] < policies["purchase_date"]) | (policies["policy_end_date"] < policies["policy_start_date"])
results.append(check(
    "Policy dates logically consistent",
    not bad_policy_dates.any(),
    f"violations={bad_policy_dates.sum():,}"
))

renewal_join = renewals.merge(
    policies[["policy_id","customer_id","policy_end_date","product_id"]],
    on=["policy_id","customer_id"], how="left"
)
bad_renewal_dates = renewal_join["renewal_date"] <= renewal_join["policy_end_date"]
results.append(check(
    "Renewal date follows policy end date",
    not bad_renewal_dates.any(),
    f"violations={bad_renewal_dates.sum():,}"
))

## 6. Customer distribution

In [ ]:
customers["registration_year"] = customers["registration_date"].dt.year
customers["registration_month"] = customers["registration_date"].dt.to_period("M").astype(str)

print("Customers by registration year")
display(customers.groupby("registration_year").size().rename("customers").to_frame())

print("\nItaly vs foreign customers")
country_summary = pd.DataFrame({
    "customers": customers.groupby(customers["country"].eq("IT").map({True:"Italy", False:"Foreign"})).size()
})
country_summary["share"] = country_summary["customers"] / len(customers)
display(country_summary)

print("\nTop Italian regions")
display(
    customers.loc[customers["country"].eq("IT")]
    .groupby("region").size()
    .sort_values(ascending=False)
    .rename("customers").head(20).to_frame()
)

print("\nAcquisition channels")
display(
    customers.groupby("acquisition_channel").size()
    .sort_values(ascending=False)
    .rename("customers").to_frame()
)

year_counts = customers.groupby("registration_year").size().to_dict()
results.append(check("2024 customer acquisition = 20,000", year_counts.get(2024,0)==20000, str(year_counts.get(2024,0))))
results.append(check("2025 customer acquisition = 70,000", year_counts.get(2025,0)==70000, str(year_counts.get(2025,0))))
results.append(check("2026 customer acquisition = 110,000", year_counts.get(2026,0)==110000, str(year_counts.get(2026,0))))

italy_share = customers["country"].eq("IT").mean()
results.append(check("Italy customer share between 90% and 95%", 0.90 <= italy_share <= 0.95, f"{italy_share:.2%}"))

## 7. Product and purchasing behaviour

In [ ]:
pol = policies.merge(products, on="product_id", how="left")

print("Policies by product")
product_mix = pol.groupby(["business_line","product_type"]).size().rename("policies").to_frame()
product_mix["share"] = product_mix["policies"] / len(pol)
display(product_mix.sort_values("policies", ascending=False))

print("\nPolicies by business line")
display(pol.groupby("business_line").size().sort_values(ascending=False).rename("policies").to_frame())

customer_line_counts = pol.groupby("customer_id")["business_line"].nunique()
print("\nNumber of business lines purchased per customer")
display(customer_line_counts.value_counts().sort_index().rename("customers").to_frame())

travel_single = pol[(pol["business_line"]=="Travel") & (pol["product_type"]=="Single Trip")]
travel_single_counts = travel_single.groupby("customer_id").size()

one_shot_customers = (travel_single_counts == 1).sum()
repeat_travel_customers = (travel_single_counts > 1).sum()

print(f"\nTravel Single Trip one-shot customers: {one_shot_customers:,}")
print(f"Travel Single Trip repeat customers:   {repeat_travel_customers:,}")

results.append(check(
    "Travel is the dominant policy business line",
    pol["business_line"].value_counts().idxmax()=="Travel",
    str(pol["business_line"].value_counts().to_dict())
))

## 8. Pricing validation

In [ ]:
print("Premium summary by product")
display(
    pol.groupby(["business_line","product_type"])["premium_amount"]
    .agg(["count","min","median","mean","max"])
)

travel = pol[pol["business_line"]=="Travel"].copy()
print("\nTravel Single Trip premium by destination")
display(
    travel[travel["product_type"]=="Single Trip"]
    .groupby("destination")["premium_amount"]
    .agg(["count","median","mean","min","max"])
    .sort_values("mean", ascending=False)
)

invalid_premiums = (policies["premium_amount"] <= 0).sum()
results.append(check("All policy premiums > 0", invalid_premiums==0, f"invalid={invalid_premiums:,}"))

pet_summary = pol[pol["business_line"]=="Pet"].groupby("product_type")["premium_amount"].mean()
if {"Standard","Premium"}.issubset(pet_summary.index):
    results.append(check("Pet Premium average > Pet Standard average", pet_summary["Premium"] > pet_summary["Standard"], str(pet_summary.to_dict())))

home_summary = pol[pol["business_line"]=="Home"].groupby("product_type")["premium_amount"].mean()
if {"Standard","Premium"}.issubset(home_summary.index):
    results.append(check("Home Premium average > Home Standard average", home_summary["Premium"] > home_summary["Standard"], str(home_summary.to_dict())))

## 9. Quote-to-policy funnel

In [ ]:
converted_quote_ids = set(policies["quote_id"])
quotes["converted"] = quotes["quote_id"].isin(converted_quote_ids)

overall_conversion = quotes["converted"].mean()
print(f"Overall quote-to-policy conversion rate: {overall_conversion:.2%}")

funnel = (
    quotes.merge(products, on="product_id", how="left")
    .groupby(["business_line","product_type"])
    .agg(quotes=("quote_id","size"), converted=("converted","sum"))
)
funnel["conversion_rate"] = funnel["converted"] / funnel["quotes"]
display(funnel.sort_values("conversion_rate", ascending=False))

channel_funnel = (
    quotes.groupby("sales_channel_id")
    .agg(quotes=("quote_id","size"), converted=("converted","sum"))
)
channel_funnel["conversion_rate"] = channel_funnel["converted"] / channel_funnel["quotes"]
display(channel_funnel.sort_values("conversion_rate", ascending=False))

results.append(check("Every policy has a valid originating quote", policies["quote_id"].isin(quotes["quote_id"]).all()))
results.append(check("Both converted and non-converted quotes exist", quotes["converted"].any() and (~quotes["converted"]).any(), f"conversion={overall_conversion:.2%}"))

## 10. Marketing attribution and channel rules

In [ ]:
policies["has_campaign"] = policies["campaign_id"].notna()

attribution = policies.groupby("sales_channel_id").agg(
    policies=("policy_id","size"),
    attributed=("has_campaign","sum")
)
attribution["attribution_rate"] = attribution["attributed"] / attribution["policies"]
display(attribution)

agency_bad = policies.loc[policies["sales_channel_id"].eq("AGENCY"), "campaign_id"].notna().sum()
partner_bad = policies.loc[policies["sales_channel_id"].eq("PARTNER"), "campaign_id"].notna().sum()
assisted_missing = policies.loc[policies["sales_channel_id"].eq("ASSISTED"), "campaign_id"].isna().sum()

results.append(check("Agency policies have no campaign attribution", agency_bad==0, f"violations={agency_bad:,}"))
results.append(check("Partner policies have no campaign attribution", partner_bad==0, f"violations={partner_bad:,}"))
results.append(check("Assisted Sales policies have campaign attribution", assisted_missing==0, f"missing={assisted_missing:,}"))

web_rates = attribution.loc["WEB","attribution_rate"] if "WEB" in attribution.index else np.nan
results.append(check("Web contains both direct and campaign-driven purchases", 0 < web_rates < 1, f"attribution={web_rates:.2%}"))

## 11. Campaign timing and seasonality

In [ ]:
campaign_products = campaigns.merge(products[["product_id","business_line","product_type"]], on="product_id", how="left", suffixes=("","_product"))

print("Campaigns by business line")
display(campaigns.groupby("business_line").size().rename("campaigns").to_frame())

travel_policies = pol[pol["business_line"]=="Travel"].copy()
travel_policies["purchase_month"] = travel_policies["purchase_date"].dt.month

print("\nTravel policies by purchase month")
travel_month = travel_policies.groupby("purchase_month").size().rename("policies").to_frame()
display(travel_month)

results.append(check(
    "Travel has more campaigns than Pet and Home",
    (campaigns["business_line"].eq("Travel").sum() > campaigns["business_line"].eq("Pet").sum()) and
    (campaigns["business_line"].eq("Travel").sum() > campaigns["business_line"].eq("Home").sum()),
    str(campaigns["business_line"].value_counts().to_dict())
))

# interaction must fall within campaign period when campaign is populated
ic = interactions.dropna(subset=["campaign_id"]).merge(
    campaigns[["campaign_id","start_date","end_date"]], on="campaign_id", how="left"
)
bad_interaction_dates = (ic["interaction_date"] < ic["start_date"]) | (ic["interaction_date"] > ic["end_date"])
results.append(check(
    "Campaign interactions occur inside campaign period",
    not bad_interaction_dates.any(),
    f"violations={bad_interaction_dates.sum():,}"
))

## 12. Renewal and churn readiness

In [ ]:
renewal_products = renewal_join.merge(
    products[["product_id","business_line","product_type"]],
    on="product_id", how="left"
)

single_trip_renewals = (
    (renewal_products["business_line"]=="Travel") &
    (renewal_products["product_type"]=="Single Trip")
).sum()

results.append(check(
    "Travel Single Trip policies have no renewals",
    single_trip_renewals==0,
    f"violations={single_trip_renewals:,}"
))

print("Renewals by product")
display(
    renewal_products.groupby(["business_line","product_type"]).size()
    .sort_values(ascending=False)
    .rename("renewals").to_frame()
)

# Churn-readiness illustration: eligible expired renewable policies vs successful renewal events
renewable = pol[
    ((pol["business_line"]=="Travel") & (pol["product_type"]=="Annual")) |
    (pol["business_line"].isin(["Pet","Home"]))
].copy()

renewed_policy_ids = set(renewals["policy_id"])
renewable["renewed_flag"] = renewable["policy_id"].isin(renewed_policy_ids)
eligible = renewable[renewable["policy_end_date"] <= pd.Timestamp("2026-12-31")]

print(f"Renewable policies expired by end of observation window: {len(eligible):,}")
print(f"Successful renewals among them: {eligible['renewed_flag'].sum():,}")
print(f"Non-renewed eligible policies: {(~eligible['renewed_flag']).sum():,}")

## 13. Final validation summary

In [ ]:
validation = pd.DataFrame(results)
display(validation)

print("\nStatus summary")
display(validation["status"].value_counts().rename("checks").to_frame())

failures = validation[validation["status"]=="FAIL"]
if failures.empty:
    print("✅ All automated validation checks passed.")
else:
    print(f"❌ {len(failures)} validation check(s) failed.")
    display(failures)

## Notes

This notebook validates the current published version of the Wossist synthetic dataset.

The validation rules are intentionally aligned with the documented business model rather than with a generic insurance data-quality framework. As the Wossist ecosystem evolves, new validation checks can be introduced alongside new datasets and analytics use cases.